# 02 — Fetch contributors per repo

For the top 500 repos by stars, pulls the top 30 contributors and saves raw JSON.

In [ ]:
import requests
import json
import time
import os
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv()
TOKEN = os.getenv("GITHUB_TOKEN")
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
BASE_URL = "https://api.github.com" 

In [ ]:
with open("../data/raw/repos_raw.json") as f:
    repos = json.load(f)

repos.sort(key=lambda x: x["stargazers_count"], reverse=True)
top_repos = repos[:500]
print(f"Will fetch contributors for {len(top_repos)} repos")

In [ ]:
def get_contributors(full_name):
    url = f"{BASE_URL}/repos/{full_name}/contributors"
    params = {"per_page": 30}
    r = requests.get(url, headers=HEADERS, params=params)
    if r.status_code == 200:
        return r.json()
    if r.status_code == 204:
        return []
    if r.status_code == 404:
        return None
    if r.status_code == 403:
        reset = int(r.headers.get("X-RateLimit-Reset", 0))
        wait = max(0, reset - int(time.time()))
        print(f"Rate limited, sleeping {wait}s")
        time.sleep(wait + 1)
        r = requests.get(url, headers=HEADERS, params=params)
        return r.json() if r.status_code == 200 else None
    return None

In [ ]:
contributors = {}
for repo in tqdm(top_repos):
    name = repo["full_name"]
    result = get_contributors(name)
    if result is not None:
        contributors[name] = result
    time.sleep(0.2)

print(f"\nGot contributors for {len(contributors)} repos")

In [ ]:
with open("../data/raw/contributors_raw.json", "w") as f:
    json.dump(contributors, f)
print("Saved to data/raw/contributors_raw.json")

In [ ]:
empty = sum(1 for v in contributors.values() if len(v) == 0)
print(f"Repos with empty contributor lists: {empty}")

sample_name = list(contributors.keys())[0]
print(f"\nSample - {sample_name}:")
for c in contributors[sample_name][:5]:
    print(f"  {c['login']}: {c['contributions']} commits")